In [0]:
# Cell 1 - Storage Configuration
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

storage_account_name = "nyctaxidatalakesa"
storage_account_key = "YOUR_ACCOUNT_STORAGE_KEY_HERE"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)
print("Storage configuration set successfully.")

Storage configuration set successfully.


In [0]:
# Cell 2 - Load month by month with safe casting
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, IntegerType
from functools import reduce

base = "abfss://nyc-taxi-raw@nyctaxidatalakesa.dfs.core.windows.net"

keep = ["VendorID","tpep_pickup_datetime","tpep_dropoff_datetime",
        "passenger_count","trip_distance","RatecodeID",
        "PULocationID","DOLocationID","payment_type",
        "fare_amount","tip_amount","total_amount"]

def load_month(yr, mo):
    path = f"{base}/yellow_tripdata_{yr}-{mo:02d}.parquet"
    df = spark.read.option("mergeSchema","false").parquet(path)
    df = df.withColumn("VendorID", col("VendorID").cast(IntegerType()))\
           .withColumn("passenger_count", col("passenger_count").cast(DoubleType()))\
           .withColumn("trip_distance", col("trip_distance").cast(DoubleType()))\
           .withColumn("RatecodeID", col("RatecodeID").cast(DoubleType()))\
           .withColumn("PULocationID", col("PULocationID").cast(IntegerType()))\
           .withColumn("DOLocationID", col("DOLocationID").cast(IntegerType()))\
           .withColumn("payment_type", col("payment_type").cast(DoubleType()))\
           .withColumn("fare_amount", col("fare_amount").cast(DoubleType()))\
           .withColumn("tip_amount", col("tip_amount").cast(DoubleType()))\
           .withColumn("total_amount", col("total_amount").cast(DoubleType()))
    avail = [c for c in keep if c in df.columns]
    return df.select(avail)

all_dfs = []
for yr in [2023, 2024]:
    for mo in range(1, 13):
        try:
            all_dfs.append(load_month(yr, mo))
            print(f"✓ {yr}-{mo:02d}")
        except Exception as e:
            print(f"✗ {yr}-{mo:02d}: {str(e)[:50]}")

df = reduce(lambda a,b: a.union(b), all_dfs)
df.cache()
print(f"\nTotal rows: {df.count():,}")
print("All files loaded successfully!")

✓ 2023-01
✓ 2023-02
✓ 2023-03
✓ 2023-04
✓ 2023-05
✓ 2023-06
✓ 2023-07
✓ 2023-08
✓ 2023-09
✓ 2023-10
✓ 2023-11
✓ 2023-12
✓ 2024-01
✓ 2024-02
✓ 2024-03
✓ 2024-04
✓ 2024-05
✓ 2024-06
✓ 2024-07
✓ 2024-08
✓ 2024-09
✓ 2024-10
✓ 2024-11
✓ 2024-12

Total rows: 79,479,946
All files loaded successfully!


In [0]:
# Cell 3 - Data Cleaning and Feature Engineering
from pyspark.sql.functions import col, year, month, hour, dayofweek

# Filter valid records
df_clean = df.filter(
    (col("fare_amount") > 0) &
    (col("fare_amount") < 500) &
    (col("trip_distance") > 0) &
    (col("trip_distance") < 100) &
    (col("passenger_count") > 0) &
    (col("passenger_count") <= 6) &
    (col("total_amount") > 0) &
    (year("tpep_pickup_datetime").isin([2023, 2024]))
).dropna(subset=["fare_amount","trip_distance",
                 "PULocationID","DOLocationID"])

# Add time features
df_clean = df_clean\
    .withColumn("pickup_year", year("tpep_pickup_datetime"))\
    .withColumn("pickup_month", month("tpep_pickup_datetime"))\
    .withColumn("pickup_hour", hour("tpep_pickup_datetime"))\
    .withColumn("pickup_dayofweek", dayofweek("tpep_pickup_datetime"))

df_clean.cache()
print(f"Rows after cleaning: {df_clean.count():,}")
print(f"Rows removed: {df.count() - df_clean.count():,}")

Rows after cleaning: 71,237,813
Rows removed: 8,242,133


In [0]:
# Cell 4 - Cleaned Dataset Summary

print("=== CLEANED DATASET OVERVIEW ===")
print(f"Rows: {df_clean.count():,}")
print(f"Columns: {len(df_clean.columns)}")

print("\n=== CLEANED SCHEMA ===")
df_clean.printSchema()

print("\n=== SAMPLE CLEANED RECORDS ===")
display(df_clean.limit(10))

print("\n=== NUMERIC SUMMARY AFTER CLEANING ===")
display(
    df_clean.select(
        "fare_amount",
        "trip_distance",
        "passenger_count",
        "total_amount",
        "pickup_year",
        "pickup_month",
        "pickup_hour",
        "pickup_dayofweek"
    ).describe()
)

=== CLEANED DATASET OVERVIEW ===
Rows: 71,237,813
Columns: 16

=== CLEANED SCHEMA ===
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickup_year: integer (nullable = true)
 |-- pickup_month: integer (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_dayofweek: integer (nullable = true)


=== SAMPLE CLEANED RECORDS ===


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,tip_amount,total_amount,pickup_year,pickup_month,pickup_hour,pickup_dayofweek
2,2023-01-01T00:32:10,2023-01-01T00:40:36,1.0,0.97,1.0,161,141,2.0,9.3,0.0,14.3,2023,1,0,1
2,2023-01-01T00:55:08,2023-01-01T01:01:27,1.0,1.1,1.0,43,237,1.0,7.9,4.0,16.9,2023,1,0,1
2,2023-01-01T00:25:04,2023-01-01T00:37:49,1.0,2.51,1.0,48,238,1.0,14.9,15.0,34.9,2023,1,0,1
2,2023-01-01T00:10:29,2023-01-01T00:21:19,1.0,1.43,1.0,107,79,1.0,11.4,3.28,19.68,2023,1,0,1
2,2023-01-01T00:50:34,2023-01-01T01:02:52,1.0,1.84,1.0,161,137,1.0,12.8,10.0,27.8,2023,1,0,1
2,2023-01-01T00:09:22,2023-01-01T00:19:49,1.0,1.66,1.0,239,143,1.0,12.1,3.42,20.52,2023,1,0,1
2,2023-01-01T00:27:12,2023-01-01T00:49:56,1.0,11.7,1.0,142,200,1.0,45.7,10.74,64.44,2023,1,0,1
2,2023-01-01T00:21:44,2023-01-01T00:36:40,1.0,2.95,1.0,164,236,1.0,17.7,5.68,28.38,2023,1,0,1
2,2023-01-01T00:39:42,2023-01-01T00:50:36,1.0,3.01,1.0,141,107,2.0,14.9,0.0,19.9,2023,1,0,1
2,2023-01-01T00:53:01,2023-01-01T01:01:45,1.0,1.8,1.0,234,68,1.0,11.4,3.28,19.68,2023,1,0,1



=== NUMERIC SUMMARY AFTER CLEANING ===


summary,fare_amount,trip_distance,passenger_count,total_amount,pickup_year,pickup_month,pickup_hour,pickup_dayofweek
count,71237813,71237813,71237813,71237813,71237813,71237813,71237813,71237813
mean,19.740834614897118,3.47279654724911,1.370077756317421,28.957635517340403,2023.500127874504,6.5707474484091755,14.34731662803854,4.119941217173525
stddev,18.062667623858747,4.563928236786732,0.8470854180384692,22.851035127641424,0.49999998720811084,3.463751374877878,5.743210632055537,1.9548877250532528
min,0.01,0.01,1.0,0.01,2023,1,0,1
max,499.3,99.92,6.0,1737.18,2024,12,23,7


In [0]:
# Cell 5 - Demand Analysis by Hour, Day and Month
from pyspark.sql.functions import col, count, avg, round

print("=== HOURLY DEMAND ===")
hourly = df_clean.groupBy("pickup_hour")\
    .agg(count("*").alias("trip_count"),
         round(avg("fare_amount"),2).alias("avg_fare"))\
    .orderBy("pickup_hour")
display(hourly)

print("=== DAY OF WEEK DEMAND ===")
# 1=Sunday, 2=Monday ... 7=Saturday
daily = df_clean.groupBy("pickup_dayofweek")\
    .agg(count("*").alias("trip_count"),
         round(avg("fare_amount"),2).alias("avg_fare"))\
    .orderBy("pickup_dayofweek")
display(daily)

print("=== MONTHLY DEMAND ===")
monthly = df_clean.groupBy("pickup_year","pickup_month")\
    .agg(count("*").alias("trip_count"),
         round(avg("fare_amount"),2).alias("avg_fare"))\
    .orderBy("pickup_year","pickup_month")
display(monthly)

=== HOURLY DEMAND ===


pickup_hour,trip_count,avg_fare
0,1951381,20.22
1,1291263,18.14
2,843197,16.82
3,544355,17.82
4,348437,23.86
5,391093,28.24
6,934727,23.2
7,1874490,19.54
8,2624696,18.6
9,3024540,18.72


Databricks visualization. Run in Databricks to view.

=== DAY OF WEEK DEMAND ===


pickup_dayofweek,trip_count,avg_fare
1,8869393,20.66
2,8996170,20.54
3,10376510,19.57
4,10849295,19.55
5,11176147,19.87
6,10565662,19.52
7,10404636,18.71


Databricks visualization. Run in Databricks to view.

=== MONTHLY DEMAND ===


pickup_year,pickup_month,trip_count,avg_fare
2023,1,2884134,18.54
2023,2,2732567,18.43
2023,3,3190542,19.12
2023,4,3077707,19.58
2023,5,3281602,20.08
2023,6,3084666,20.05
2023,7,2712793,19.98
2023,8,2628718,20.01
2023,9,2604687,20.75
2023,10,3243616,20.32


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Cell 6 - Top Pickup and Dropoff Zones
from pyspark.sql.functions import count, col

print("=== TOP 10 PICKUP ZONES ===")
top_pickup = df_clean.groupBy("PULocationID")\
    .agg(count("*").alias("pickup_count"))\
    .orderBy(col("pickup_count").desc())\
    .limit(10)
display(top_pickup)

print("=== TOP 10 DROPOFF ZONES ===")
top_dropoff = df_clean.groupBy("DOLocationID")\
    .agg(count("*").alias("dropoff_count"))\
    .orderBy(col("dropoff_count").desc())\
    .limit(10)
display(top_dropoff)

=== TOP 10 PICKUP ZONES ===


PULocationID,pickup_count
132,3747865
237,3469278
161,3413912
236,3059207
162,2601167
186,2515208
138,2508745
230,2448998
142,2364187
163,2114673


=== TOP 10 DROPOFF ZONES ===


DOLocationID,dropoff_count
236,3226691
237,3094711
161,2765529
230,2248991
170,2133309
162,2072523
142,2027421
239,2013307
141,1903640
68,1827227


In [0]:
## Cell 7 - Join with Taxi Zone Lookup Table
from pyspark.sql.functions import col, count

zone_lookup = spark.read.option("header","true")\
    .csv("abfss://nyc-taxi-raw@nyctaxidatalakesa.dfs.core.windows.net/taxi_zone_lookup.csv")

zone_lookup = zone_lookup.select("LocationID","Borough","Zone")

# Join top pickup zones with names
top_pickup_named = df_clean.groupBy("PULocationID")\
    .agg(count("*").alias("pickup_count"))\
    .orderBy(col("pickup_count").desc())\
    .limit(10)\
    .join(zone_lookup, 
          df_clean.PULocationID.cast("string") == zone_lookup.LocationID, 
          "left")\
    .select("PULocationID","Zone","Borough","pickup_count")

display(top_pickup_named)

# Also top dropoff zones
top_dropoff_named = df_clean.groupBy("DOLocationID")\
    .agg(count("*").alias("dropoff_count"))\
    .orderBy(col("dropoff_count").desc())\
    .limit(10)\
    .join(zone_lookup,
          df_clean.DOLocationID.cast("string") == zone_lookup.LocationID,
          "left")\
    .select("DOLocationID","Zone","Borough","dropoff_count")

display(top_dropoff_named)

PULocationID,Zone,Borough,pickup_count
132,JFK Airport,Queens,3747865
237,Upper East Side South,Manhattan,3469278
161,Midtown Center,Manhattan,3413912
236,Upper East Side North,Manhattan,3059207
162,Midtown East,Manhattan,2601167
186,Penn Station/Madison Sq West,Manhattan,2515208
138,LaGuardia Airport,Queens,2508745
230,Times Sq/Theatre District,Manhattan,2448998
142,Lincoln Square East,Manhattan,2364187
163,Midtown North,Manhattan,2114673


Databricks visualization. Run in Databricks to view.

DOLocationID,Zone,Borough,dropoff_count
236,Upper East Side North,Manhattan,3226691
237,Upper East Side South,Manhattan,3094711
161,Midtown Center,Manhattan,2765529
230,Times Sq/Theatre District,Manhattan,2248991
170,Murray Hill,Manhattan,2133309
162,Midtown East,Manhattan,2072523
142,Lincoln Square East,Manhattan,2027421
239,Upper West Side South,Manhattan,2013307
141,Lenox Hill West,Manhattan,1903640
68,East Chelsea,Manhattan,1827227


Databricks visualization. Run in Databricks to view.

In [0]:
# Cell 8 - Year over Year Comparison
from pyspark.sql.functions import count, avg, round

yoy = df_clean.groupBy("pickup_year")\
    .agg(
        count("*").alias("total_trips"),
        round(avg("fare_amount"),2).alias("avg_fare"),
        round(avg("trip_distance"),2).alias("avg_distance"),
        round(avg("tip_amount"),2).alias("avg_tip"),
        round(avg("total_amount"),2).alias("avg_total")
    ).orderBy("pickup_year")
display(yoy)

pickup_year,total_trips,avg_fare,avg_distance,avg_tip,avg_total
2023,35609797,19.73,3.51,3.6,28.91
2024,35628016,19.76,3.44,3.66,29.01


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Cell 9 - Fare and Distance Distribution
print("=== FARE DISTRIBUTION ===")
fare_dist = df_clean.groupBy(
    (col("fare_amount") - col("fare_amount") % 5).cast("int").alias("fare_bucket")
).count().orderBy("fare_bucket").limit(20)
display(fare_dist)

print("=== DISTANCE DISTRIBUTION ===")
dist_dist = df_clean.groupBy(
    col("trip_distance").cast("int").alias("distance_miles")
).count().orderBy("distance_miles").limit(20)
display(dist_dist)

=== FARE DISTRIBUTION ===


fare_bucket,count
0,993606
5,18745663
10,21128056
15,10164258
20,5485472
25,3199718
30,2112054
35,1604421
40,1373516
45,1130668


Databricks visualization. Run in Databricks to view.

=== DISTANCE DISTRIBUTION ===


distance_miles,count
0,15743802
1,23840329
2,11369133
3,5236004
4,2696588
5,1699229
6,1199492
7,943850
8,1077233
9,1155814


Databricks visualization. Run in Databricks to view.

In [0]:
# Cell 10 - Anomaly Detection
from pyspark.sql.functions import col, count, avg, stddev, round

stats = df_clean.agg(
    avg("fare_amount").alias("mean_fare"),
    stddev("fare_amount").alias("std_fare"),
    avg("trip_distance").alias("mean_dist"),
    stddev("trip_distance").alias("std_dist")
).collect()[0]

mean_fare = stats["mean_fare"]
std_fare = stats["std_fare"]
mean_dist = stats["mean_dist"]
std_dist = stats["std_dist"]

anomalies = df_clean.filter(
    (col("fare_amount") > mean_fare + 3 * std_fare) |
    (col("fare_amount") < mean_fare - 3 * std_fare) |
    (col("trip_distance") > mean_dist + 3 * std_dist)
)

print(f"Total anomalies detected: {anomalies.count():,}")
print(f"Anomaly rate: {anomalies.count()/df_clean.count()*100:.2f}%")
display(anomalies.limit(20))

Total anomalies detected: 3,005,492
Anomaly rate: 4.22%


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,tip_amount,total_amount,pickup_year,pickup_month,pickup_hour,pickup_dayofweek
1,2023-01-01T00:13:30,2023-01-01T00:44:00,1.0,17.8,2.0,132,116,1.0,70.0,15.85,95.15,2023,1,0,1
1,2023-01-01T00:21:15,2023-01-01T00:52:25,1.0,19.2,1.0,132,25,1.0,72.3,15.2,91.25,2023,1,0,1
2,2023-01-01T00:02:19,2023-01-01T00:30:49,1.0,20.37,2.0,132,140,1.0,70.0,12.0,92.55,2023,1,0,1
2,2023-01-01T00:29:33,2023-01-01T01:08:08,1.0,13.73,4.0,132,265,1.0,82.8,21.32,107.87,2023,1,0,1
1,2023-01-01T00:18:33,2023-01-01T00:54:44,1.0,21.0,2.0,132,238,1.0,70.0,0.0,81.8,2023,1,0,1
1,2023-01-01T00:21:56,2023-01-01T00:50:39,1.0,17.6,2.0,132,141,1.0,70.0,10.0,85.25,2023,1,0,1
1,2023-01-01T00:49:25,2023-01-01T01:18:52,4.0,18.9,2.0,132,45,1.0,70.0,0.0,75.25,2023,1,0,1
2,2023-01-01T00:01:55,2023-01-01T00:36:32,2.0,21.4,2.0,132,211,1.0,70.0,11.29,86.54,2023,1,0,1
2,2023-01-01T00:26:07,2023-01-01T00:58:34,1.0,19.19,1.0,132,97,1.0,73.0,15.35,92.1,2023,1,0,1
2,2023-01-01T00:51:35,2023-01-01T01:31:37,1.0,21.53,1.0,132,40,1.0,82.8,17.0,103.55,2023,1,0,1


In [0]:
# Cell 11 - ML Fare Prediction using Linear Regression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

# Prepare clean ML dataset
ml_df = df_clean.select(
    col("fare_amount").alias("label"),
    col("trip_distance"),
    col("pickup_hour"),
    col("pickup_dayofweek"),
    col("pickup_month")
).dropna()

# Sample to speed up
ml_df = ml_df.sample(0.1, seed=42)

assembler = VectorAssembler(
    inputCols=["trip_distance","pickup_hour",
               "pickup_dayofweek","pickup_month"],
    outputCol="features",
    handleInvalid="skip"
)

ml_df = assembler.transform(ml_df).select("features","label")
train, test = ml_df.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(featuresCol="features", labelCol="label")
model = lr.fit(train)

predictions = model.transform(test)
evaluator = RegressionEvaluator(labelCol="label",
    predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
r2_eval = RegressionEvaluator(labelCol="label",
    predictionCol="prediction", metricName="r2")
r2 = r2_eval.evaluate(predictions)

print(f"Linear Regression Results:")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.4f}")
print(f"Coefficients: {model.coefficients}")
display(predictions.select("label","prediction").limit(20))

Uploading artifacts:   0%|          | 0/4 [00:00<?, ?it/s]

Linear Regression Results:
RMSE: 6.11
R2 Score: 0.8852
Coefficients: [3.7231912196288146,0.019063459579881533,0.10089317710007877,0.12757985799336805]


label,prediction
3.0,6.157860179263278
3.0,5.590628035822569
3.0,5.60969149540245
3.0,6.2341140175828045
55.0,6.089518042122291
3.0,5.705008793301858
3.0,6.0076883246020945
3.0,6.228538138382134
3.0,5.781262631621384
250.0,5.781262631621384


In [0]:
# Cell 12 - Save Cleaned Data to Processed Container
df_clean.write\
    .mode("overwrite")\
    .parquet("abfss://nyc-taxi-processed@nyctaxidatalakesa.dfs.core.windows.net/cleaned_taxi_data/")

print("Cleaned data saved to nyc-taxi-processed container successfully!")
print(f"Total rows saved: {df_clean.count():,}")

Cleaned data saved to nyc-taxi-processed container successfully!
Total rows saved: 71,237,813
